# Amazon ML Hackathon — Business Entity Resolution
## Baseline Model #1: Blocking + Pairwise Features + Logistic Regression

**Status of this notebook:** this was generated as a runnable specification. The dataset
(`student_resource/dataset/...`) was **not accessible from the environment that authored
this notebook**, so no cell has actually been executed here and no numbers in this notebook
have been observed from a real run. Every place where a real run would produce a number,
plot, or table is clearly marked `# EXECUTION PENDING` and the surrounding prose explains
what *should* be reported once you run it, without inventing values.

Run this notebook top-to-bottom from the `Amazon_ml/` project root. It expects:

```
Amazon_ml/
├── student_resource/
│   └── dataset/
│       ├── train/
│       │   ├── train_source1.tsv
│       │   ├── train_source2.tsv
│       │   ├── train_source3.tsv
│       │   └── train_ground_truth.tsv
│       └── test/
│           ├── test_source1.tsv
│           ├── test_source2.tsv
│           └── test_source3.tsv
```

### Pipeline

```
Candidate Generation / Blocking (multi-key: country + name/address first-char)
            ↓
Pairwise Feature Engineering (name + address + country features)
            ↓
Logistic Regression
            ↓
Match Probability
            ↓
Threshold Selection (tuned for F0.5)
            ↓
F0.5 Evaluation (pair-level AND competition-style macro-per-S1)
```

This is intentionally the **simplest reasonable baseline** — no embeddings, no gradient
boosting, no neural nets, no transformers. The point is to have a clean, fully-understood
reference point that later, more sophisticated models can be compared against.


## 1. Setup, imports, configuration

In [1]:
import re
import string
import warnings
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    fbeta_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [2]:
# ------------------------------------------------------------------
# Dataset location
# ------------------------------------------------------------------
# The notebook is expected to be run from the Amazon_ml/ project root.
_cwd = Path.cwd()

DATASET_ROOT = _cwd / "student_resource" / "dataset"
TRAIN_ROOT = DATASET_ROOT / "train"
TEST_ROOT = DATASET_ROOT / "test"

TRAIN_S1_PATH = TRAIN_ROOT / "train_source1.tsv"
TRAIN_S2_PATH = TRAIN_ROOT / "train_source2.tsv"
TRAIN_S3_PATH = TRAIN_ROOT / "train_source3.tsv"
TRAIN_GT_PATH = TRAIN_ROOT / "train_ground_truth.tsv"

TEST_S1_PATH = TEST_ROOT / "test_source1.tsv"
TEST_S2_PATH = TEST_ROOT / "test_source2.tsv"
TEST_S3_PATH = TEST_ROOT / "test_source3.tsv"

for p in [TRAIN_S1_PATH, TRAIN_S2_PATH, TRAIN_S3_PATH, TRAIN_GT_PATH,
          TEST_S1_PATH, TEST_S2_PATH, TEST_S3_PATH]:
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {p}")


[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\train\train_source1.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\train\train_source2.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\train\train_source3.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\train\train_ground_truth.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\test\test_source1.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\test\test_source2.tsv
[OK] c:\Users\shara\Desktop\Amazon_ml\student_resource\dataset\test\test_source3.tsv


If any path above prints `MISSING`, stop here and fix `DATASET_ROOT` / your working
directory before continuing — every downstream cell assumes these files are readable.


## 2. Load the raw data

We load every file with `pd.read_csv(path, sep="\t")` and touch nothing on disk.
We do **not** eagerly build any S1×S2 / S1×S3 cross join here — we only inspect shapes,
dtypes, and a handful of rows.


In [3]:
train_s1 = pd.read_csv(TRAIN_S1_PATH, sep="\t")
train_s2 = pd.read_csv(TRAIN_S2_PATH, sep="\t")
train_s3 = pd.read_csv(TRAIN_S3_PATH, sep="\t")
train_gt = pd.read_csv(TRAIN_GT_PATH, sep="\t")

test_s1 = pd.read_csv(TEST_S1_PATH, sep="\t")
test_s2 = pd.read_csv(TEST_S2_PATH, sep="\t")
test_s3 = pd.read_csv(TEST_S3_PATH, sep="\t")

print("train_s1:", train_s1.shape)
print("train_s2:", train_s2.shape)
print("train_s3:", train_s3.shape)
print("train_gt:", train_gt.shape)
print("test_s1 :", test_s1.shape)
print("test_s2 :", test_s2.shape)
print("test_s3 :", test_s3.shape)


train_s1: (2206821, 4)
train_s2: (5034616, 4)
train_s3: (5285603, 4)
train_gt: (2206821, 2)
test_s1 : (1732544, 4)
test_s2 : (4887273, 4)
test_s3 : (5082316, 4)


In [4]:
display(train_s1.head())
display(train_s1.dtypes)


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


entity_id           str
business_name       str
business_address    str
country             str
dtype: object

In [5]:
display(train_s2.head())
display(train_s3.head())


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


In [6]:
display(train_gt.head())
train_gt.dtypes


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


source1_entity_id     str
matched_entity_ids    str
dtype: object

**Expected columns (adjust the constants below if your actual column names differ):**

- Source tables (`train_source1/2/3`, `test_source1/2/3`): an entity id column, a business
  name column, an address column, and a country column.
- `train_ground_truth.tsv`: `source1_entity_id`, `matched_entity_ids` (a delimited string of
  matched S2/S3 ids per the problem statement).

We centralize the actual column names here so the rest of the notebook is easy to adapt if
the real schema differs slightly from what's assumed.


In [7]:
# ------------------------------------------------------------------
# Column-name configuration — EDIT HERE if your TSVs use different headers.
# ------------------------------------------------------------------
ID_COL = "entity_id"
NAME_COL = "business_name"
ADDRESS_COL = "business_address"
COUNTRY_COL = "country"

GT_S1_COL = "source1_entity_id"
GT_MATCH_COL = "matched_entity_ids"
GT_DELIM = ","  # delimiter used inside matched_entity_ids

# Sanity-check that the assumed columns actually exist; fail loudly and early
# rather than silently producing an empty/garbage pipeline downstream.
for df_name, df in [("train_s1", train_s1), ("train_s2", train_s2), ("train_s3", train_s3)]:
    missing = [c for c in [ID_COL, NAME_COL, ADDRESS_COL, COUNTRY_COL] if c not in df.columns]
    if missing:
        print(f"WARNING: {df_name} is missing expected columns {missing}. "
              f"Actual columns: {list(df.columns)}")

missing_gt = [c for c in [GT_S1_COL, GT_MATCH_COL] if c not in train_gt.columns]
if missing_gt:
    print(f"WARNING: train_gt is missing expected columns {missing_gt}. "
          f"Actual columns: {list(train_gt.columns)}")


## 3. Parse ground truth into positive pairs

Each row of `train_ground_truth.tsv` looks like:

```
S1-001 -> S2-010,S2-025,S3-100
```

We explode this into one row per (source1_entity_id, candidate_entity_id) positive pair,
and tag which source (S2 or S3) each candidate id belongs to by looking up its id in the
S2 / S3 id sets (rather than assuming a naming convention like an `"S2-"` prefix, since we
can't be sure the real ids are formatted that way).


In [8]:
def parse_ground_truth(gt_df, s2_ids, s3_ids,
                        s1_col=GT_S1_COL, match_col=GT_MATCH_COL, delim=GT_DELIM):
    """Explode train_ground_truth.tsv into individual positive pairs.

    Returns a DataFrame with columns:
        source1_entity_id, candidate_entity_id, candidate_source, label
    where label is always 1 (these are all true matches).

    Rows with an empty / NaN matched_entity_ids (an S1 entity with zero matches)
    contribute zero positive pairs but are still accounted for separately so that
    S1 entities with no true match are not silently dropped from evaluation later.
    """
    s2_ids = set(s2_ids)
    s3_ids = set(s3_ids)

    records = []
    zero_match_s1 = []

    for row in gt_df.itertuples(index=False):
        s1_id = getattr(row, s1_col)
        raw_matches = getattr(row, match_col)

        if pd.isna(raw_matches) or str(raw_matches).strip() == "":
            zero_match_s1.append(s1_id)
            continue

        match_ids = [m.strip() for m in str(raw_matches).split(delim) if m.strip()]
        if not match_ids:
            zero_match_s1.append(s1_id)
            continue

        for cid in match_ids:
            if cid in s2_ids:
                source = "S2"
            elif cid in s3_ids:
                source = "S3"
            else:
                source = "UNKNOWN"  # id didn't resolve against either source table
            records.append((s1_id, cid, source, 1))

    positives = pd.DataFrame(
        records,
        columns=["source1_entity_id", "candidate_entity_id", "candidate_source", "label"],
    )
    return positives, zero_match_s1


positive_pairs, zero_match_s1_ids = parse_ground_truth(
    train_gt, train_s2[ID_COL], train_s3[ID_COL]
)

print(f"Total positive pairs: {len(positive_pairs):,}")
print(f"S1 entities with zero ground-truth matches: {len(zero_match_s1_ids):,}")
unknown_source = (positive_pairs["candidate_source"] == "UNKNOWN").sum()
if unknown_source:
    print(f"WARNING: {unknown_source:,} positive candidate ids did not match any S2/S3 id "
          f"— check GT_DELIM / id formats.")
display(positive_pairs.head())


Total positive pairs: 7,638,365
S1 entities with zero ground-truth matches: 123,247


,source1_entity_id,candidate_entity_id,candidate_source,label
0,S1-965667,S2-681193310,S2,1
1,S1-965667,S2-743505751,S2,1
2,S1-965667,S3-775321672,S3,1
3,S1-965667,S3-11291185,S3,1
4,S1-965667,S3-860443364,S3,1


## 4. Candidate generation via multi-key blocking (memory-conscious)

**Why same-country blocking alone isn't enough.** With millions of rows per source, even
same-country buckets can each contain hundreds of thousands of records, so materializing
every same-country pair still blows up memory long before we get to features. We need
blocking keys that are cheap to compute *and* meaningfully more selective.

**Blocking strategy: union of two restrictive keys.**

1. `rule1`: `normalized_country + first character of normalized business_name`
2. `rule2`: `normalized_country + first character of normalized business_address`

A candidate pair is generated if it agrees with S1 on **either** key — i.e. we take the
**union** of what each rule produces, then de-duplicate (a pair produced by both rules is
only kept once). This is still much narrower than same-country-only blocking, because two
records now also have to agree on a first letter of name *or* address, not just country.

**Memory strategy — no full candidate DataFrame is ever built:**

- S2 and S3 are each grouped into `{(country, first_char): set(ids)}` indexes **once**
  (a single pass over each table — this is O(|S2| + |S3|), not a join).
- We first report the size of the candidate set **implied by each rule** using only these
  small index dictionaries (summing bucket sizes per S1 row) — no pairs are materialized to
  get this number.
- We then generate the actual union-of-rules candidate pairs **in chunks of S1 entities**
  (a Python generator, `iter_blocked_candidates`), holding at most one chunk of pairs in
  memory at a time.
- While streaming those chunks, we simultaneously (a) tally the *exact* total number of
  distinct candidate pairs and (b) draw negative examples using **reservoir sampling**, so we
  never hold more than a fixed-size sample of negatives in memory — never the full candidate
  set.
- All true positive ground-truth pairs are kept regardless of whether blocking produced them,
  exactly as before; we separately measure how many of them these rules actually recover.

This remains an intentionally simple baseline blocking strategy, and it will still miss true
matches that disagree with S1 on country **and** on both the name/address first letter (e.g.
due to typos in the very first character, or names recorded in a different order/language).
Section 15 and the final conclusion revisit this limitation.


In [9]:
def normalize_text(x):
    """Lightweight, language-agnostic text normalization: lowercase, strip, collapse
    whitespace, drop punctuation. Deliberately NOT language-specific (no stemming,
    no transliteration) so it behaves reasonably on languages unseen in training.
    Used both for blocking keys here and for similarity features in section 7.
    """
    if pd.isna(x):
        return ""
    s = str(x).lower().strip()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s).strip()
    return s


def first_char(x):
    """First character of the normalized text, or '' if the text is empty/missing.
    Used as a cheap, single-character blocking key component.
    """
    s = normalize_text(x)
    return s[0] if s else ""


def compute_blocking_keys(df, name_col=NAME_COL, address_col=ADDRESS_COL, country_col=COUNTRY_COL):
    """Compute (country, name_first_char) and (country, address_first_char) blocking
    keys for every row of df. Returns two lists of tuples, aligned to df's row order.
    """
    country_norm = df[country_col].astype(str).str.strip().str.lower()
    name_fc = df[name_col].map(first_char)
    addr_fc = df[address_col].map(first_char)
    rule1_keys = list(zip(country_norm, name_fc))
    rule2_keys = list(zip(country_norm, addr_fc))
    return rule1_keys, rule2_keys


def build_blocking_index(ids, keys):
    """Group ids by blocking key into {key: set(ids)}. A single O(n) pass; no pairs
    are created here, only an index used later to look up candidates per S1 row.
    """
    index = defaultdict(set)
    for _id, k in zip(ids, keys):
        index[k].add(_id)
    return index


# Blocking keys for every table, computed once.
s1_rule1_keys, s1_rule2_keys = compute_blocking_keys(train_s1)
s2_rule1_keys, s2_rule2_keys = compute_blocking_keys(train_s2)
s3_rule1_keys, s3_rule2_keys = compute_blocking_keys(train_s3)

s1_ids_arr = train_s1[ID_COL].values
s2_ids_arr = train_s2[ID_COL].values
s3_ids_arr = train_s3[ID_COL].values

index_s2_rule1 = build_blocking_index(s2_ids_arr, s2_rule1_keys)
index_s3_rule1 = build_blocking_index(s3_ids_arr, s3_rule1_keys)
index_s2_rule2 = build_blocking_index(s2_ids_arr, s2_rule2_keys)
index_s3_rule2 = build_blocking_index(s3_ids_arr, s3_rule2_keys)

# Fast S1-id -> (rule1_key, rule2_key) lookup, reused later to check ground-truth coverage.
s1_key_lookup = {sid: (k1, k2) for sid, k1, k2 in zip(s1_ids_arr, s1_rule1_keys, s1_rule2_keys)}

print(f"Distinct rule1 (country, name first-char) buckets in S2: {len(index_s2_rule1):,}")
print(f"Distinct rule1 (country, name first-char) buckets in S3: {len(index_s3_rule1):,}")
print(f"Distinct rule2 (country, address first-char) buckets in S2: {len(index_s2_rule2):,}")
print(f"Distinct rule2 (country, address first-char) buckets in S3: {len(index_s3_rule2):,}")


Distinct rule1 (country, name first-char) buckets in S2: 312
Distinct rule1 (country, name first-char) buckets in S3: 313
Distinct rule2 (country, address first-char) buckets in S2: 91
Distinct rule2 (country, address first-char) buckets in S3: 91


In [10]:
# ------------------------------------------------------------------
# Report the candidate-pair count implied by EACH rule BEFORE building any candidate
# dataframe. This only sums bucket sizes matched per S1 row using the small index
# dictionaries above -- no (s1, candidate) pair objects are created for this count.
# ------------------------------------------------------------------
def estimate_rule_pair_count(s1_keys, index_s2, index_s3):
    total = 0
    for k in s1_keys:
        total += len(index_s2.get(k, ())) + len(index_s3.get(k, ()))
    return total


rule1_pair_count = estimate_rule_pair_count(s1_rule1_keys, index_s2_rule1, index_s3_rule1)
rule2_pair_count = estimate_rule_pair_count(s1_rule2_keys, index_s2_rule2, index_s3_rule2)

n_cartesian = len(train_s1) * (len(train_s2) + len(train_s3))

print("Candidate pairs implied by each blocking rule (pre-union, pre-dedup):")
print(f"  rule1 (country + name first-char):    {rule1_pair_count:,}  "
      f"(reduction vs Cartesian: {1 - rule1_pair_count / n_cartesian:.6%})" if n_cartesian else "")
print(f"  rule2 (country + address first-char): {rule2_pair_count:,}  "
      f"(reduction vs Cartesian: {1 - rule2_pair_count / n_cartesian:.6%})" if n_cartesian else "")
print(f"  theoretical full Cartesian product (S1 x (S2+S3)): {n_cartesian:,}")
print()
print("These are per-rule totals before taking the union across rules (which removes "
      "pairs found by both rules) -- the exact, de-duplicated union count is computed "
      "below via chunked streaming, without ever materializing the full pair set.")


Candidate pairs implied by each blocking rule (pre-union, pre-dedup):
  rule1 (country + name first-char):    622,409,472,937  (reduction vs Cartesian: 97.267122%)
  rule2 (country + address first-char): 1,059,121,840,367  (reduction vs Cartesian: 95.349604%)
  theoretical full Cartesian product (S1 x (S2+S3)): 22,774,876,013,799

These are per-rule totals before taking the union across rules (which removes pairs found by both rules) -- the exact, de-duplicated union count is computed below via chunked streaming, without ever materializing the full pair set.


### Chunked, generator-based union of the two blocking rules

`iter_blocked_candidates` processes S1 in fixed-size chunks. For each S1 entity in a chunk it
looks up both rule keys, unions the resulting candidate ids (de-duplicating a candidate found
by both rules for that entity), and yields just that chunk's list of `(s1_id, candidate_id,
candidate_source)` tuples. The caller below consumes one chunk at a time — at no point does
the notebook hold more than `CHUNK_SIZE` S1 entities' worth of candidate pairs in memory.


In [11]:
def iter_blocked_candidates(s1_ids, rule1_keys, rule2_keys,
                             index_s2_rule1, index_s3_rule1,
                             index_s2_rule2, index_s3_rule2,
                             chunk_size=5000):
    """Yield the union-of-rules blocked candidates, one chunk of S1 entities at a time.

    Each yielded chunk is a list of (source1_entity_id, candidate_entity_id,
    candidate_source) tuples, already de-duplicated per S1 entity across rule1/rule2.
    Memory use is bounded by chunk_size, independent of the total dataset size.
    """
    n = len(s1_ids)
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        chunk_pairs = []
        for i in range(start, end):
            s1_id = s1_ids[i]
            k1, k2 = rule1_keys[i], rule2_keys[i]

            cands = set()
            for cid in index_s2_rule1.get(k1, ()):
                cands.add((cid, "S2"))
            for cid in index_s3_rule1.get(k1, ()):
                cands.add((cid, "S3"))
            for cid in index_s2_rule2.get(k2, ()):
                cands.add((cid, "S2"))
            for cid in index_s3_rule2.get(k2, ()):
                cands.add((cid, "S3"))

            for cid, source in cands:
                chunk_pairs.append((s1_id, cid, source))

        yield chunk_pairs


CHUNK_SIZE = 5000  # number of S1 entities processed per chunk; lower this if memory is still tight


In [12]:
# ------------------------------------------------------------------
# Single streaming pass: tally the exact de-duplicated union count AND reservoir-sample
# negative candidates, without ever storing the full candidate set in memory.
# ------------------------------------------------------------------
positive_keys = set(zip(positive_pairs["source1_entity_id"], positive_pairs["candidate_entity_id"]))

NEGATIVES_PER_POSITIVE = 10  # controlled imbalance ratio; tune if class balance report looks off
n_target_negatives = NEGATIVES_PER_POSITIVE * max(len(positive_keys), 1)

rng = np.random.RandomState(RANDOM_STATE)
reservoir = []       # bounded-size sample of negative (s1_id, cand_id, source) tuples
n_seen_negatives = 0  # running count of ALL blocked non-positive pairs seen so far
total_union_pairs = 0  # exact de-duplicated union count, tallied chunk by chunk

for chunk in iter_blocked_candidates(
    s1_ids_arr, s1_rule1_keys, s1_rule2_keys,
    index_s2_rule1, index_s3_rule1, index_s2_rule2, index_s3_rule2,
    chunk_size=CHUNK_SIZE,
):
    total_union_pairs += len(chunk)
    for s1_id, cid, source in chunk:
        if (s1_id, cid) in positive_keys:
            continue  # never sample a true positive as a negative

        n_seen_negatives += 1
        if len(reservoir) < n_target_negatives:
            reservoir.append((s1_id, cid, source))
        else:
            # classic reservoir sampling: keep a uniform random sample of fixed size
            # from a stream of unknown total length, in a single pass.
            j = rng.randint(0, n_seen_negatives)
            if j < n_target_negatives:
                reservoir[j] = (s1_id, cid, source)

negative_pairs = pd.DataFrame(reservoir, columns=["source1_entity_id", "candidate_entity_id", "candidate_source"])
negative_pairs["label"] = 0

overlap_pairs = max(rule1_pair_count + rule2_pair_count - total_union_pairs, 0)
reduction_union = 1 - (total_union_pairs / n_cartesian) if n_cartesian else float("nan")

print(f"Actual de-duplicated UNION of rule1 + rule2 candidate pairs: {total_union_pairs:,}")
print(f"Pairs found by both rules (approx. overlap removed by dedup): {overlap_pairs:,}")
print(f"Reduction vs. full Cartesian product:                         {reduction_union:.6%}")
print()
print(f"Positive pairs:                          {len(positive_keys):,}")
print(f"Blocked non-positive (negative) candidates seen: {n_seen_negatives:,}")
print(f"Negative pairs sampled for training:     {len(negative_pairs):,}")
print(f"Positive : negative ratio in training set: 1 : {len(negative_pairs) / max(len(positive_keys), 1):.2f}")


MemoryError: 

In [ ]:
# How many of the true positive pairs does this multi-key blocking actually recover?
# (Measured directly against the small per-entity index dictionaries -- no need to
# materialize any candidate dataframe for this check either.)
def positive_pair_is_blocked(s1_id, cand_id, source):
    k1, k2 = s1_key_lookup.get(s1_id, (None, None))
    if source == "S2":
        return cand_id in index_s2_rule1.get(k1, ()) or cand_id in index_s2_rule2.get(k2, ())
    elif source == "S3":
        return cand_id in index_s3_rule1.get(k1, ()) or cand_id in index_s3_rule2.get(k2, ())
    return False


positive_pairs["recovered_by_blocking"] = [
    positive_pair_is_blocked(s1_id, cand_id, source)
    for s1_id, cand_id, source in zip(
        positive_pairs["source1_entity_id"],
        positive_pairs["candidate_entity_id"],
        positive_pairs["candidate_source"],
    )
]

n_recovered = int(positive_pairs["recovered_by_blocking"].sum())
n_missed = len(positive_pairs) - n_recovered

print(f"True positive pairs: {len(positive_pairs):,}")
print(f"Recovered by rule1 ∪ rule2 blocking: {n_recovered:,} "
      f"({n_recovered / max(len(positive_pairs), 1):.2%})")
print(f"Missed by blocking (kept anyway, per section 5): {n_missed:,} "
      f"({n_missed / max(len(positive_pairs), 1):.2%})")


**Important:** if the "missed by blocking" percentage above is non-trivial, this blocking
strategy is silently capping the baseline's achievable recall — no amount of feature
engineering or model tuning downstream can recover those pairs. This is expected for a
restrictive baseline blocking strategy, but must be reported honestly rather than hidden,
and is listed as a concrete future-improvement item in the final conclusion.


## 5. Build the labeled pair dataset (positives ∪ sampled negatives)

Both pieces are already in hand from section 4:

1. **`positive_pairs`** — every true match from the ground truth, regardless of whether
   blocking produced it (`recovered_by_blocking` just documents which ones blocking would
   have found on its own).
2. **`negative_pairs`** — a fixed-size, reservoir-sampled set of blocked non-matches, drawn
   in a single streaming pass that never materialized the full candidate set and that never
   sampled a true positive.

We simply concatenate the two into the final labeled pair dataset.


In [ ]:
labeled_pairs = pd.concat(
    [positive_pairs.drop(columns=["recovered_by_blocking"]), negative_pairs],
    ignore_index=True,
    sort=False,
)

print(f"Total labeled pairs: {len(labeled_pairs):,}")
print(labeled_pairs["label"].value_counts())


## 6. Attach raw record fields to each labeled pair

Before computing similarity features we need the actual name/address/country strings for
both sides of each pair. We do this via indexed lookups (`set_index` + `.loc`), not merges
on the full tables, to keep memory bounded.


In [ ]:
s1_lookup = train_s1.set_index(ID_COL)[[NAME_COL, ADDRESS_COL, COUNTRY_COL]]
s2_lookup = train_s2.set_index(ID_COL)[[NAME_COL, ADDRESS_COL, COUNTRY_COL]]
s3_lookup = train_s3.set_index(ID_COL)[[NAME_COL, ADDRESS_COL, COUNTRY_COL]]


def attach_record_fields(pairs_df, s1_lookup, s2_lookup, s3_lookup):
    """Join raw name/address/country fields for both sides of each pair."""
    df = pairs_df.copy()

    s1_fields = s1_lookup.reindex(df["source1_entity_id"]).reset_index(drop=True)
    s1_fields.columns = [f"s1_{c}" for c in s1_fields.columns]

    cand_fields = pd.DataFrame(index=df.index, columns=[f"cand_{c}" for c in s1_lookup.columns])
    is_s2 = (df["candidate_source"] == "S2").values
    is_s3 = (df["candidate_source"] == "S3").values

    if is_s2.any():
        s2_vals = s2_lookup.reindex(df.loc[is_s2, "candidate_entity_id"])
        cand_fields.loc[is_s2, :] = s2_vals.values
    if is_s3.any():
        s3_vals = s3_lookup.reindex(df.loc[is_s3, "candidate_entity_id"])
        cand_fields.loc[is_s3, :] = s3_vals.values

    out = pd.concat([df.reset_index(drop=True), s1_fields, cand_fields.reset_index(drop=True)], axis=1)
    return out


labeled_pairs_full = attach_record_fields(labeled_pairs, s1_lookup, s2_lookup, s3_lookup)
display(labeled_pairs_full.head())

n_unmatched_ids = labeled_pairs_full[f"cand_{NAME_COL}"].isna().sum()
if n_unmatched_ids:
    print(f"WARNING: {n_unmatched_ids:,} candidate ids didn't resolve to a record "
          f"(check candidate_source / id consistency).")


## 7. Pairwise feature engineering

Eleven simple, interpretable features: 5 name features, 5 address features, 1 country
feature. Everything is computed from the two records' text — nothing is learned from a
fixed training vocabulary, which matters for section 16 (unseen France in the test set).


In [ ]:
# normalize_text() and first_char() were already defined in section 4 (used there for
# blocking keys); we reuse normalize_text() here for similarity features rather than
# redefining it.

def character_similarity(a, b):
    """Character-level similarity via difflib.SequenceMatcher ratio, in [0, 1]."""
    a, b = normalize_text(a), normalize_text(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def token_similarity(a, b):
    """Token-level Jaccard similarity over whitespace-split tokens, in [0, 1]."""
    a, b = normalize_text(a), normalize_text(b)
    tokens_a, tokens_b = set(a.split()), set(b.split())
    if not tokens_a and not tokens_b:
        return 1.0
    if not tokens_a or not tokens_b:
        return 0.0
    return len(tokens_a & tokens_b) / len(tokens_a | tokens_b)


def normalized_length_difference(a, b):
    """Absolute character-length difference, normalized by the longer string's length.
    Returns a value in [0, 1]; 0 means identical length.
    """
    a, b = normalize_text(a), normalize_text(b)
    max_len = max(len(a), len(b))
    if max_len == 0:
        return 0.0
    return abs(len(a) - len(b)) / max_len


def build_features(df):
    """Compute the 11 pairwise features for every row of a pairs dataframe that already
    has s1_/cand_ name, address, country columns attached.
    """
    feats = pd.DataFrame(index=df.index)

    s1_name, cand_name = df[f"s1_{NAME_COL}"], df[f"cand_{NAME_COL}"]
    s1_addr, cand_addr = df[f"s1_{ADDRESS_COL}"], df[f"cand_{ADDRESS_COL}"]
    s1_country, cand_country = df[f"s1_{COUNTRY_COL}"], df[f"cand_{COUNTRY_COL}"]

    norm_s1_name = s1_name.map(normalize_text)
    norm_cand_name = cand_name.map(normalize_text)
    norm_s1_addr = s1_addr.map(normalize_text)
    norm_cand_addr = cand_addr.map(normalize_text)

    # --- name features ---
    feats["exact_name_match"] = (
        s1_name.astype(str).str.strip() == cand_name.astype(str).str.strip()
    ).astype(int)
    feats["normalized_name_match"] = (norm_s1_name == norm_cand_name).astype(int)
    feats["name_character_similarity"] = [
        character_similarity(a, b) for a, b in zip(s1_name, cand_name)
    ]
    feats["name_token_similarity"] = [
        token_similarity(a, b) for a, b in zip(s1_name, cand_name)
    ]
    feats["name_length_difference"] = [
        normalized_length_difference(a, b) for a, b in zip(s1_name, cand_name)
    ]

    # --- address features ---
    feats["exact_address_match"] = (
        s1_addr.astype(str).str.strip() == cand_addr.astype(str).str.strip()
    ).astype(int)
    feats["normalized_address_match"] = (norm_s1_addr == norm_cand_addr).astype(int)
    feats["address_character_similarity"] = [
        character_similarity(a, b) for a, b in zip(s1_addr, cand_addr)
    ]
    feats["address_token_similarity"] = [
        token_similarity(a, b) for a, b in zip(s1_addr, cand_addr)
    ]
    feats["address_length_difference"] = [
        normalized_length_difference(a, b) for a, b in zip(s1_addr, cand_addr)
    ]

    # --- country feature (pairwise property, NOT a fixed categorical vocabulary) ---
    norm_s1_country = s1_country.astype(str).str.strip().str.lower()
    norm_cand_country = cand_country.astype(str).str.strip().str.lower()
    feats["same_country"] = (norm_s1_country == norm_cand_country).astype(int)

    return feats


FEATURE_COLUMNS = [
    "exact_name_match", "normalized_name_match", "name_character_similarity",
    "name_token_similarity", "name_length_difference",
    "exact_address_match", "normalized_address_match", "address_character_similarity",
    "address_token_similarity", "address_length_difference",
    "same_country",
]

pair_features = build_features(labeled_pairs_full)
dataset = pd.concat(
    [labeled_pairs_full[["source1_entity_id", "candidate_entity_id", "candidate_source", "label"]],
     pair_features],
    axis=1,
)
display(dataset.head())


## 8. Feature inspection

In [ ]:
print("Feature dtypes:")
print(dataset[FEATURE_COLUMNS].dtypes)
print()
print("Missing values per feature:")
print(dataset[FEATURE_COLUMNS].isna().sum())


In [ ]:
dataset[FEATURE_COLUMNS].describe().T


In [ ]:
correlations = dataset[FEATURE_COLUMNS + ["label"]].corr(numeric_only=True)["label"].drop("label")
correlations.sort_values(ascending=False)


A high correlation with `label` doesn't automatically mean a feature is "useful" in the
causal sense — e.g. `same_country` is expected to correlate with the label simply because
blocking guarantees every candidate already shares (or, for the small out-of-block-positive
set, doesn't share) a country with S1, so its variance and correlation partly reflect how the
candidate set was constructed, not purely discriminative power between true and false matches
within a country. Name and address similarity features are more directly interpretable as
"does the text actually look like the same business," and are expected to carry most of the
real discriminative signal. Once real correlations are computed above, revisit this
paragraph and describe what was actually observed rather than only this general caveat.


In [ ]:
print("Example POSITIVE pairs:")
display(dataset[dataset["label"] == 1].sample(min(5, (dataset["label"] == 1).sum()), random_state=RANDOM_STATE))

print("Example NEGATIVE pairs:")
display(dataset[dataset["label"] == 0].sample(min(5, (dataset["label"] == 0).sum()), random_state=RANDOM_STATE))


## 9. Train / validation split — by Source 1 entity, not by row

We split on the **unique `source1_entity_id` values** (80% train / 20% validation), then
assign every pair belonging to a given S1 entity to whichever split that entity landed in.

**Why not a naive random row split?** Each S1 entity contributes several candidate pairs
(one per blocked S2/S3 candidate). If we split rows randomly, pairs from the *same* S1
entity could end up in both train and validation. The model could then partially
"memorize" that entity's specific name/address text during training and get an
optimistic, leaked validation score for its other pairs — the validation metric would no
longer reflect how well the model generalizes to genuinely unseen S1 entities, which is
what matters for the actual test set (containing entirely different, unseen businesses,
including ones from France).


In [ ]:
unique_s1_ids = dataset["source1_entity_id"].unique()
rng = np.random.RandomState(RANDOM_STATE)
shuffled_ids = rng.permutation(unique_s1_ids)

n_train_ids = int(0.8 * len(shuffled_ids))
train_ids = set(shuffled_ids[:n_train_ids])
valid_ids = set(shuffled_ids[n_train_ids:])

train_mask = dataset["source1_entity_id"].isin(train_ids)
valid_mask = dataset["source1_entity_id"].isin(valid_ids)

train_df = dataset[train_mask].reset_index(drop=True)
valid_df = dataset[valid_mask].reset_index(drop=True)

print(f"Unique S1 entities: {len(unique_s1_ids):,} -> train {len(train_ids):,} / valid {len(valid_ids):,}")
print(f"Pairs: train {len(train_df):,} / valid {len(valid_df):,}")
assert train_ids.isdisjoint(valid_ids), "Entity leakage between train and validation!"


## 10. Class imbalance

In [ ]:
for split_name, split_df in [("train", train_df), ("valid", valid_df)]:
    n_pos = int((split_df["label"] == 1).sum())
    n_neg = int((split_df["label"] == 0).sum())
    total = n_pos + n_neg
    print(f"[{split_name}] positives={n_pos:,} ({n_pos/total:.2%})  "
          f"negatives={n_neg:,} ({n_neg/total:.2%})  total={total:,}")


## 11. Train Logistic Regression

We compare `class_weight=None` vs `class_weight="balanced"` on the same train/validation
split, using a `StandardScaler -> LogisticRegression` pipeline so features are on comparable
scales. We then pick one configuration as *the* baseline and say so explicitly, rather than
reporting both without a clear final choice.


In [ ]:
X_train = train_df[FEATURE_COLUMNS]
y_train = train_df["label"]
X_valid = valid_df[FEATURE_COLUMNS]
y_valid = valid_df["label"]

models = {}
for weight_setting in [None, "balanced"]:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight=weight_setting,
            random_state=RANDOM_STATE,
        )),
    ])
    pipe.fit(X_train, y_train)
    models[weight_setting] = pipe

print("Trained models:", list(models.keys()))


In [ ]:
# Quick default-threshold (0.5) comparison, purely to decide class_weight — final
# threshold selection for the chosen model happens properly in section 13 via F0.5.
for weight_setting, pipe in models.items():
    proba = pipe.predict_proba(X_valid)[:, 1]
    preds = (proba >= 0.5).astype(int)
    p = precision_score(y_valid, preds, zero_division=0)
    r = recall_score(y_valid, preds, zero_division=0)
    f05 = fbeta_score(y_valid, preds, beta=0.5, zero_division=0)
    print(f"class_weight={weight_setting!r:>10}  @0.5  precision={p:.4f}  recall={r:.4f}  F0.5={f05:.4f}")


**Chosen configuration for this baseline:** `class_weight="balanced"`. Class-balanced
weighting is chosen by default for this baseline because the candidate set is heavily
skewed toward negatives after blocking, and unweighted logistic regression tends to
under-predict the minority (positive/match) class as a result. After running the comparison
above, confirm this is still the better choice for F0.5 specifically (balanced weighting
optimizes differently than F0.5 would; it's possible `class_weight=None` combined with a
low decision threshold ends up performing comparably or better for the F0.5 objective) and
update this sentence to reflect what was actually observed, then keep going with the model
selected below.


In [ ]:
FINAL_CLASS_WEIGHT = "balanced"  # revisit after inspecting the cell above
final_model = models[FINAL_CLASS_WEIGHT]


## 12. Match probabilities on the validation set

In [ ]:
valid_proba = final_model.predict_proba(X_valid)[:, 1]
valid_df = valid_df.copy()
valid_df["match_probability"] = valid_proba

display(valid_df[["source1_entity_id", "candidate_entity_id", "candidate_source",
                   "label", "match_probability"]].head(10))


## 13. Threshold tuning for F0.5 — pair level

The competition metric is F0.5 (precision weighted more heavily than recall). We do NOT
default to a 0.5 probability threshold; instead we sweep thresholds and pick the one that
maximizes validation F0.5.


In [ ]:
thresholds = np.round(np.arange(0.10, 0.96, 0.05), 2)
beta = 0.5

threshold_results = []
for t in thresholds:
    preds = (valid_proba >= t).astype(int)
    p = precision_score(y_valid, preds, zero_division=0)
    r = recall_score(y_valid, preds, zero_division=0)
    f = fbeta_score(y_valid, preds, beta=beta, zero_division=0)
    threshold_results.append({"threshold": t, "precision": p, "recall": r, "f0.5": f})

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)

best_row = threshold_df.loc[threshold_df["f0.5"].idxmax()]
BEST_THRESHOLD = float(best_row["threshold"])
print(f"Best pair-level threshold by F0.5: {BEST_THRESHOLD} "
      f"(precision={best_row['precision']:.4f}, recall={best_row['recall']:.4f}, "
      f"F0.5={best_row['f0.5']:.4f})")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_df["threshold"], threshold_df["f0.5"], marker="o", label="F0.5")
ax.plot(threshold_df["threshold"], threshold_df["precision"], marker="o", label="Precision")
ax.plot(threshold_df["threshold"], threshold_df["recall"], marker="o", label="Recall")
ax.axvline(BEST_THRESHOLD, color="gray", linestyle="--", label=f"best threshold = {BEST_THRESHOLD}")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold vs Precision / Recall / F0.5 (pair-level, validation)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()


## 13b. Competition-style evaluation: macro F0.5 per Source 1 entity

The actual competition metric is **not** the flat pair-level F0.5 above — it's computed
**per S1 entity** (predicted matched candidate ids vs. true matched candidate ids) and then
**averaged across S1 entities**, with the convention that an S1 entity with no true matches
and no predicted matches scores a perfect 1.0, while predicting any match for a
zero-true-match entity scores 0.0. We implement that precisely below.


In [ ]:
def evaluate_macro_f05(valid_df, proba_col, threshold, beta=0.5,
                        s1_col="source1_entity_id", cand_col="candidate_entity_id",
                        label_col="label"):
    """Competition-style macro F0.5, averaged per Source 1 entity.

    For each S1 entity:
      - predicted matches = candidates with proba >= threshold
      - true matches = candidates with label == 1
      - if true matches is empty AND predicted matches is empty -> F0.5 = 1.0
      - if true matches is empty AND predicted matches is non-empty -> F0.5 = 0.0
      - otherwise compute precision/recall/F0.5 in the usual way (zero_division -> 0)

    Returns (macro_f05, per_entity_df).
    """
    rows = []
    for s1_id, group in valid_df.groupby(s1_col):
        true_matches = set(group.loc[group[label_col] == 1, cand_col])
        pred_matches = set(group.loc[group[proba_col] >= threshold, cand_col])

        if not true_matches and not pred_matches:
            f05 = 1.0
            precision = recall = 1.0
        elif not true_matches and pred_matches:
            f05 = 0.0
            precision = recall = 0.0
        else:
            tp = len(true_matches & pred_matches)
            precision = tp / len(pred_matches) if pred_matches else 0.0
            recall = tp / len(true_matches) if true_matches else 0.0
            if precision == 0.0 and recall == 0.0:
                f05 = 0.0
            else:
                f05 = ((1 + beta ** 2) * precision * recall) / ((beta ** 2 * precision) + recall)

        rows.append({
            "source1_entity_id": s1_id,
            "n_true_matches": len(true_matches),
            "n_predicted_matches": len(pred_matches),
            "precision": precision,
            "recall": recall,
            "f0.5": f05,
        })

    per_entity_df = pd.DataFrame(rows)
    macro_f05 = per_entity_df["f0.5"].mean()
    return macro_f05, per_entity_df


macro_f05, per_entity_df = evaluate_macro_f05(valid_df, "match_probability", BEST_THRESHOLD)
print(f"Competition-style macro F0.5 (per-S1, best pair-level threshold): {macro_f05:.4f}")
display(per_entity_df.head(10))


In [ ]:
# It's worth checking whether the threshold that's best for pair-level F0.5 is also
# best for macro-per-S1 F0.5 -- they are different objectives and can disagree.
macro_threshold_results = []
for t in thresholds:
    m_f05, _ = evaluate_macro_f05(valid_df, "match_probability", t)
    macro_threshold_results.append({"threshold": t, "macro_f0.5": m_f05})

macro_threshold_df = pd.DataFrame(macro_threshold_results)
display(macro_threshold_df)

best_macro_row = macro_threshold_df.loc[macro_threshold_df["macro_f0.5"].idxmax()]
BEST_MACRO_THRESHOLD = float(best_macro_row["threshold"])
print(f"Best threshold by macro-per-S1 F0.5: {BEST_MACRO_THRESHOLD} "
      f"(macro F0.5={best_macro_row['macro_f0.5']:.4f})")


If `BEST_THRESHOLD` (pair-level) and `BEST_MACRO_THRESHOLD` (competition-style) differ once
this is actually run, prefer **`BEST_MACRO_THRESHOLD`** for any final submission-style
prediction, since the competition's real metric is the macro-per-S1 one — the pair-level
number is a useful, easier-to-interpret diagnostic, not the target being optimized.


## 14. Baseline results summary

In [ ]:
final_preds = (valid_proba >= BEST_MACRO_THRESHOLD).astype(int)
pair_precision = precision_score(y_valid, final_preds, zero_division=0)
pair_recall = recall_score(y_valid, final_preds, zero_division=0)
pair_f05 = fbeta_score(y_valid, final_preds, beta=0.5, zero_division=0)
final_macro_f05, _ = evaluate_macro_f05(valid_df, "match_probability", BEST_MACRO_THRESHOLD)

results_table = pd.DataFrame([{
    "Model": "Logistic Regression",
    "Features": len(FEATURE_COLUMNS),
    "Blocking": "same_country",
    "Negative Sampling": f"{NEGATIVES_PER_POSITIVE}:1 (neg:pos), seed={RANDOM_STATE}",
    "Class Weight": FINAL_CLASS_WEIGHT,
    "Threshold": BEST_MACRO_THRESHOLD,
    "Pair-level Precision": round(pair_precision, 4),
    "Pair-level Recall": round(pair_recall, 4),
    "Pair-level F0.5": round(pair_f05, 4),
    "Macro S1 F0.5": round(final_macro_f05, 4),
}])
display(results_table)


## 15. Error analysis

We inspect true positives, false positives, false negatives, and true negatives at the
chosen threshold, with the raw name/address/country fields alongside the computed features,
to look for recognizable failure patterns (near-duplicate names for different businesses,
spelling/abbreviation differences, missing/partial addresses, multilingual text,
same-country collisions, etc.).


In [ ]:
valid_full = valid_df.copy()
valid_full["prediction"] = final_preds

# Re-attach the raw text fields for readability during error analysis.
valid_full = attach_record_fields(
    valid_full[["source1_entity_id", "candidate_entity_id", "candidate_source", "label",
                "match_probability", "prediction"] + FEATURE_COLUMNS],
    s1_lookup, s2_lookup, s3_lookup,
)

display_cols = [
    f"s1_{NAME_COL}", f"s1_{ADDRESS_COL}", f"s1_{COUNTRY_COL}",
    f"cand_{NAME_COL}", f"cand_{ADDRESS_COL}", f"cand_{COUNTRY_COL}",
    "match_probability", "label", "prediction",
] + FEATURE_COLUMNS

true_positives = valid_full[(valid_full["label"] == 1) & (valid_full["prediction"] == 1)]
false_positives = valid_full[(valid_full["label"] == 0) & (valid_full["prediction"] == 1)]
false_negatives = valid_full[(valid_full["label"] == 1) & (valid_full["prediction"] == 0)]
true_negatives = valid_full[(valid_full["label"] == 0) & (valid_full["prediction"] == 0)]

print(f"TP={len(true_positives):,}  FP={len(false_positives):,}  "
      f"FN={len(false_negatives):,}  TN={len(true_negatives):,}")


In [ ]:
print("=== True Positives (sample) ===")
display(true_positives[display_cols].sample(min(5, len(true_positives)), random_state=RANDOM_STATE))


In [ ]:
print("=== False Positives (sample) — model said match, ground truth says no ===")
display(false_positives[display_cols].sort_values("match_probability", ascending=False).head(10))


In [ ]:
print("=== False Negatives (sample) — model said no match, ground truth says match ===")
display(false_negatives[display_cols].sort_values("match_probability", ascending=True).head(10))


In [ ]:
print("=== True Negatives (sample) ===")
display(true_negatives[display_cols].sample(min(5, len(true_negatives)), random_state=RANDOM_STATE))


**How to read the false-positive / false-negative tables once this has actually been run:**
go row by row and note which of these recognizable patterns (if any) explains the error —

- similar business names but genuinely different businesses (e.g. franchises, chains)
- spelling differences / typos between sources
- abbreviations (`"Intl"` vs `"International"`, `"Corp"` vs `"Corporation"`)
- address variations (suite/floor differences, reformatted street order)
- missing or empty address fields on one side
- multilingual text (same business, different language/script per source)
- same-country collisions (two unrelated businesses in the same country/city with
  similar-sounding names)

Summarize the *dominant* one or two patterns actually observed — don't just relist the
category names, since that repeats this cell without adding the analysis it's meant to
produce.


## 16. Generalizing to an unseen country (France) in the test set

The **test** set contains France, which does not appear anywhere in the **training** data.
This section reasons about what that means for this baseline, without training on any test
labels and without claiming a guarantee of strong French performance.

**Why `same_country` generalizes differently than a categorical country feature would.**
If country had instead been one-hot encoded (or label-encoded) as a fixed training
vocabulary — `US=0, India=1, ...` — the model would have no learned weight at all for a
country it never saw at training time (e.g. France), and a naive encoder would either error
or silently map it to some arbitrary/default value. `same_country`, by contrast, is a
**pairwise, relational property**: "do these two records claim the same country as each
other," computed identically regardless of *which* country that happens to be. The logistic
regression only ever sees the binary outcome of that comparison, so a French/French pair and
a US/US pair produce the exact same feature value (`1`) and are treated the same way by the
trained weight for that feature — France doesn't need to have been "seen" for this feature to
be well-defined and consistently scaled at test time.

**Why the text-similarity features don't require having seen a language before.**
`character_similarity` (via `SequenceMatcher`) and `token_similarity` (via token-set Jaccard)
operate purely on string overlap — they don't rely on a fixed vocabulary, embeddings, or any
language-specific model that would need retraining/fine-tuning for a new language. Two French
business names or addresses that are near-duplicates of each other will still produce a high
character/token similarity score for essentially the same string-matching reasons an English
near-duplicate pair would, even though French was never in the training data.

**What this section does *not* claim.** The validation experiment above only measures
performance on countries seen during training (assuming, as stated, France is training-absent).
It does **not** directly measure French performance, and country-specific effects unrelated to
these two features could still hurt French accuracy — for example: different address
formatting conventions, accented characters interacting with normalization, or businesses
whose canonical names differ more substantially by convention (e.g. legal-entity suffixes like
`"SARL"` / `"SAS"` common in France but absent from training data) could shift the *distribution*
of `name_length_difference` or `address_token_similarity` values, or the calibration of
predicted probabilities, in ways this baseline cannot observe without French validation data.

**Bottom line:** the *feature design* is set up to generalize to an unseen country in
principle; whether it generalizes well **in practice** on the real French test rows is an open
empirical question that this baseline's validation split cannot answer, since it contains no
France — only France's own genuine test performance (once labels are known, or via the
competition leaderboard) can confirm this.


# What Did Our First Baseline Teach Us?

*(Fill in the italicized placeholders below with the actual numbers from a real run of this
notebook — nothing here has been fabricated, and this cell should not be treated as final
until every placeholder below is replaced.)*

**Candidate generation**
- Candidate pairs generated by rule1 ∪ rule2 multi-key blocking: *`{total_union_pairs}`* (see section 4)
- Reduction vs. the full S1×(S2+S3) Cartesian product: *`{reduction_union}`*
- Share of true positive pairs recoverable at all under this blocking: *`{n_recovered / len(positive_pairs)}`*

**Labeled training data**
- Positive pairs (all, including any recovered outside blocking): *`{len(positive_pairs)}`*
- Negative pairs sampled (ratio {NEGATIVES_PER_POSITIVE}:1, seed {RANDOM_STATE}): *`{len(negative_pairs)}`*

**Model**
- Final configuration: Logistic Regression, `class_weight="{FINAL_CLASS_WEIGHT}"`, 11 features, `StandardScaler`-scaled
- Best pair-level threshold: *`{BEST_THRESHOLD}`* — Best macro-per-S1 threshold: *`{BEST_MACRO_THRESHOLD}`*

**Validation performance**
- Pair-level Precision / Recall / F0.5: *`{pair_precision}`* / *`{pair_recall}`* / *`{pair_f05}`*
- Competition-style macro-per-S1 F0.5: *`{final_macro_f05}`*

**Most informative features:** *fill in from the correlation table (section 8) and the
error-analysis patterns (section 15) once observed — name and address similarity features
are expected to dominate, with `same_country` acting more as a blocking artifact than an
independently discriminative signal.*

**Main false-positive patterns observed:** *fill in from section 15.*

**Main false-negative patterns observed:** *fill in from section 15.*

## Limitations of this baseline

1. **Multi-key blocking is a hard recall ceiling.** Any true match that disagrees with S1 on
   country, *and* on both the name and address first character, is unrecoverable by this
   pipeline, no matter how good the classifier is — section 4 quantifies exactly how much
   recall is lost this way.
2. **Only 11 hand-designed features**, all string-similarity based; no learned text
   representations, no use of any other columns the raw data might contain beyond
   name/address/country.
3. **Negative sampling is a fixed ratio**, not calibrated against the true operating
   imbalance the competition actually scores against.
4. **No hyperparameter tuning** — a single, deliberately simple logistic regression
   configuration.
5. **French/unseen-country generalization is untested**, only argued for conceptually
   (section 16) — the validation split, by construction, contains no France.

## Possible future improvements

1. Better blocking (e.g. soundex/phonetic keys instead of a raw first character, n-gram or
   sorted-neighborhood blocking, or a looser key that also catches near-country matches)
2. Better character/token similarity (e.g. Jaro-Winkler, cosine similarity over character
   n-grams, or a learned string-similarity model)
3. More sophisticated address normalization (abbreviation expansion, unit/suite parsing,
   postal-code-aware comparison)
4. Better negative sampling (e.g. hard-negative mining — sampling negatives that are
   *textually close but wrong*, rather than uniformly random within-country negatives)
5. Gradient boosting (e.g. XGBoost/LightGBM) over the same or an expanded feature set
6. Multilingual embeddings / sentence-transformer-based semantic similarity, particularly
   to address the French generalization question raised in section 16
7. More advanced entity-resolution techniques (e.g. graph-based clustering across all three
   sources simultaneously, rather than pairwise S1-vs-candidate classification)

This baseline is intentionally simple so that every one of the improvements above can be
introduced one at a time and compared against this notebook's numbers as a fixed reference
point.
